# ============================================================
# BENCHMARK METHOD 7: RAR Thuần (Randomized AutoRegressive Modeling)
# Paper: Yu et al. (NeurIPS 2024 / ByteDance / arXiv:2411.00776)
# Flat Vocabulary N = 16,384 | Randomness Annealing r(t) : 1.0 -> 0.0
# Pure Float32 Precision + Gradient Checkpointing (Zero NaN, Zero OOM)
# ============================================================


In [ ]:
# ============================================================
# CELL 1 — Thiết lập môi trường & Clone các Repositories cần thiết
# ============================================================
import subprocess, os, sys, shutil, torch

WORK_DIR          = '/kaggle/working'
LLAMA_DIR         = f'{WORK_DIR}/LlamaGen'
MASC_DIR          = f'{WORK_DIR}/masc-upgrade-ann'
BENCHMARK_DIR     = f'{WORK_DIR}/Benchmark'
WEIGHTS_DIR       = f'{WORK_DIR}/weights'
CODES_DIR         = f'{WORK_DIR}/codes'
CKPT_DIR          = f'{WORK_DIR}/checkpoints_rar_baseline'

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(CODES_DIR, exist_ok=True)

print('Cài đặt dependencies cho RAR Baseline & LlamaGen Evaluator...')
subprocess.run('pip install -q omegaconf timm safetensors einops huggingface_hub pandas tabulate scipy', shell=True)

def clone_or_pull(url, target):
    if not os.path.exists(target):
        print(f'>>> git clone {url} {target}')
        subprocess.run(f'git clone {url} {target}', shell=True, check=True)
    else:
        print(f'>>> {target} đã tồn tại, git pull cập nhật...')
        subprocess.run(f'git -C {target} pull', shell=True)

clone_or_pull('https://github.com/FoundationVision/LlamaGen.git', LLAMA_DIR)
clone_or_pull('https://github.com/nta2112/masc-upgrade-ann.git', MASC_DIR)
clone_or_pull('https://github.com/nta2112/Masce-baseline-benchmark.git', BENCHMARK_DIR)

print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
print(f'Số lượng GPU    : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'   GPU {i} : {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)')
print('=' * 60)


In [ ]:
# ============================================================
# CELL 2 — Tải Weights Pretrained của VQ-VAE & LlamaGen GPT Transformer
# ============================================================
from huggingface_hub import hf_hub_download
import os

VQ_PATH = os.path.join(WEIGHTS_DIR, 'vq_ds16_c2i.pt')
if not os.path.exists(VQ_PATH):
    print('Đang tải Tokenizer vq_ds16_c2i.pt...')
    hf_hub_download(repo_id='FoundationVision/LlamaGen', filename='vq_ds16_c2i.pt', local_dir=WEIGHTS_DIR)
    print('✅ Đã tải: vq_ds16_c2i.pt')
else:
    print('✅ Tokenizer vq_ds16_c2i.pt đã có sẵn.')

GPT_CKPT_NAME = 'c2i_B_256.pt'
GPT_CKPT_PATH = os.path.join(WEIGHTS_DIR, GPT_CKPT_NAME)
if not os.path.exists(GPT_CKPT_PATH):
    print(f'Đang tải Pretrained weights ({GPT_CKPT_NAME})...')
    hf_hub_download(repo_id='FoundationVision/LlamaGen', filename=GPT_CKPT_NAME, local_dir=WEIGHTS_DIR)
    print(f'✅ Đã tải: {GPT_CKPT_NAME}')
else:
    print(f'✅ Pretrained checkpoint đã có sẵn: {GPT_CKPT_PATH}')


In [ ]:
# ============================================================
# CELL 3 — Chuẩn bị Dữ liệu 10-Class ImageNet & Reference Batch 1,000 Ảnh
# ============================================================
import os, sys, glob, shutil, tarfile, io
from pathlib import Path
import numpy as np
import torch
from huggingface_hub import snapshot_download

WORK_DIR     = '/kaggle/working'
CODES_DIR    = f'{WORK_DIR}/codes'
WEIGHTS_DIR  = f'{WORK_DIR}/weights'
LLAMA_DIR    = f'{WORK_DIR}/LlamaGen'
REF_NPZ      = f'{WORK_DIR}/ref_batch_1000.npz'
labels_path  = f'{CODES_DIR}/labels.npy'

# Kiểm tra nếu dataset đã có sẵn từ Kaggle Input
input_dirs = glob.glob('/kaggle/input/**/shard_00000.npy', recursive=True)
if input_dirs and not os.path.exists(labels_path):
    src_dir = os.path.dirname(input_dirs[0])
    if os.path.exists(f'{src_dir}/labels.npy'):
        print(f'Copying pre-extracted dataset from {src_dir}...')
        for f in glob.glob(f'{src_dir}/*.npy'):
            shutil.copy(f, CODES_DIR)

existing_npy = sorted(Path(CODES_DIR).glob('shard_*.npy'))
if not (existing_npy and os.path.exists(labels_path) and len(np.load(labels_path)) == 10000):
    WDS_DIR = f'{WORK_DIR}/wds_cache'
    os.makedirs(WDS_DIR, exist_ok=True)
    os.makedirs(CODES_DIR, exist_ok=True)

    tar_files = sorted(Path('/kaggle/input').glob('**/*.tar'))
    if not tar_files:
        print('Đang tải imagenet-llamagen-cache từ HuggingFace (repo_type=dataset)...')
        snapshot_download(
            repo_id='Efficient-Large-Model/imagenet-llamagen-cache',
            repo_type='dataset',
            local_dir=WDS_DIR,
            ignore_patterns=['*.json', '*.yaml', 'README*'],
        )
        tar_files = sorted(Path(WDS_DIR).glob('**/*.tar'))

    assert len(tar_files) > 0, 'Không tìm thấy file tar của dataset!'
    tar_path = tar_files[0]
    print(f'Đang trích xuất 10 class từ {tar_path.name}...')
    TARGET_SYNSETS = {
        'n01440764': 0, 'n01443537': 1, 'n01484850': 2, 'n01491361': 3, 'n01494475': 4,
        'n01496331': 5, 'n01498041': 6, 'n01514668': 7, 'n01514859': 8, 'n01518878': 9,
    }
    MAX_PER_CLASS = 1000
    class_counts = {c: 0 for c in range(10)}
    shard_codes, all_labels = [], []
    shard_idx, total_imgs = 0, 0
    SHARD_SIZE = 512

    with tarfile.open(tar_path, 'r:*') as tf:
        for member in tf:
            if not member.name.endswith('.npz'): continue
            parts = member.name.split('/')
            if len(parts) < 2: continue
            synset = parts[-2]
            label = TARGET_SYNSETS.get(synset, -1)
            if label == -1 or class_counts[label] >= MAX_PER_CLASS: continue

            f = tf.extractfile(member)
            if f is None: continue
            try:
                npz = np.load(io.BytesIO(f.read()))
                codes = npz['indices'].flatten().astype(np.int32)
            except Exception: continue
            if len(codes) != 256: continue

            shard_codes.append(codes)
            all_labels.append(label)
            class_counts[label] += 1
            total_imgs += 1

            if len(shard_codes) >= SHARD_SIZE:
                np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
                shard_codes = []; shard_idx += 1
            if total_imgs >= 10000: break

    if shard_codes:
        np.save(f'{CODES_DIR}/shard_{shard_idx:05d}.npy', np.array(shard_codes, dtype=np.int32))
    np.save(labels_path, np.array(all_labels, dtype=np.int32))
    shutil.rmtree(WDS_DIR, ignore_errors=True)
    print(f'✅ Đã chuẩn bị 10,000 samples token codes trong {CODES_DIR}!')
else:
    print(f'✅ Đã có sẵn 10,000 samples token codes trong {CODES_DIR}!')

if not os.path.exists(REF_NPZ):
    print('Đang tạo Reference Batch 10-Class CÂN BẰNG (100 ảnh/class x 10 class = 1,000 ảnh)...')
    if LLAMA_DIR not in sys.path: sys.path.insert(0, LLAMA_DIR)
    from tokenizer.tokenizer_image.vq_model import VQ_models
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

    vq = VQ_models['VQ-16'](codebook_size=16384, codebook_embed_dim=8)
    vq_ckpt = os.path.join(WEIGHTS_DIR, 'vq_ds16_c2i.pt')
    st = torch.load(vq_ckpt, map_location='cpu')
    vq.load_state_dict(st.get('model', st))
    vq.to(device).eval()

    shards = sorted(Path(CODES_DIR).glob('shard_*.npy'))
    all_codes = np.concatenate([np.load(f) for f in shards], axis=0)
    all_labels = np.load(labels_path)

    ref_indices = []
    for c in range(10):
        c_idxs = np.where(all_labels == c)[0][:100]
        ref_indices.extend(c_idxs)
    ref_indices = np.array(ref_indices)
    ref_codes = all_codes[ref_indices]

    ref_imgs = []
    bs = 20
    with torch.no_grad():
        for i in range(0, len(ref_codes), bs):
            b_c = torch.tensor(ref_codes[i:i+bs], device=device, dtype=torch.long)
            imgs = vq.decode_code(b_c, [len(b_c), 8, 16, 16])
            imgs = ((imgs.clamp(-1, 1) + 1.0) / 2.0 * 255.0).permute(0, 2, 3, 1).to(torch.uint8).cpu().numpy()
            ref_imgs.append(imgs)

    ref_arr = np.concatenate(ref_imgs, axis=0)
    np.savez_compressed(REF_NPZ, arr_0=ref_arr)
    print(f'✅ Reference Batch hoàn thành: {REF_NPZ} ({ref_arr.shape})')
    del vq; torch.cuda.empty_cache()
else:
    print(f'✅ Reference Batch đã có sẵn: {REF_NPZ}')


In [ ]:
# ============================================================
# CELL 4 — NGUYÊN LÝ RAR THUẦN (RANDOMIZED AUTOREGRESSIVE MODELING)
# ============================================================
print('=' * 70)
print('📌 RAR Thuần (Randomized AutoRegressive Baseline - Yu et al., NeurIPS 2024):')
print('   - Từ vựng (Vocabulary): N = 16,384 (Codebook đầy đủ)')
print('   - Randomness Annealing Schedule: r(t) = max(0.0, 1.0 - step / total_steps)')
print('   - Tần số RoPE vị trí 2D: token_freqs = base_freqs[order_tensor]')
print('   - Tại bước suy luận: r = 0 -> Tự động chuyển về chuẩn raster-scan')
print('=' * 70)


In [ ]:
# ============================================================
# CELL 5 — HUẤN LUYỆN 10 EPOCHS CHO RAR THUẦN (FLAT VOCABULARY N=16,384)
# Cấu hình chuẩn vàng: Pure Float32 + Gradient Checkpointing (BS=8x2x2 = 32, 3,120 steps)
# ============================================================
import subprocess, os, sys, shutil, glob

WORK_DIR          = '/kaggle/working'
LLAMA_DIR         = f'{WORK_DIR}/LlamaGen'
BENCHMARK_DIR     = f'{WORK_DIR}/Benchmark'
WEIGHTS_DIR       = f'{WORK_DIR}/weights'
CODES_DIR         = f'{WORK_DIR}/codes'
CKPT_DIR          = f'{WORK_DIR}/checkpoints_rar_baseline'
GPT_CKPT_PATH     = f'{WEIGHTS_DIR}/c2i_B_256.pt'

# Đường dẫn checkpoint người dùng cung cấp nếu có
USER_CKPT_PATH = ''

CKPT_CANDIDATES = [
    USER_CKPT_PATH,
    f'{CKPT_DIR}/ckpt_rar_baseline_10class_final.pt',
] + glob.glob('/kaggle/input/**/ckpt_rar_baseline_10class_final.pt', recursive=True)

CKPT_PATH = next((p for p in CKPT_CANDIDATES if p and os.path.exists(p) and os.path.getsize(p) > 10000000), f'{CKPT_DIR}/ckpt_rar_baseline_10class_final.pt')

if os.path.exists(CKPT_PATH):
    print('=' * 70)
    print(f'✅ ĐÃ CÓ SẴN CHECKPOINT: {CKPT_PATH}')
    print(f'   -> Dung lượng: {os.path.getsize(CKPT_PATH) / 1e6:.1f} MB')
    print('=' * 70)
    print('⏩ BỎ QUA HUẤN LUYỆN! Chuyển thẳng sang CELL 6 để sinh ảnh.')
else:
    TRAIN_SCRIPT = f'{BENCHMARK_DIR}/baselines/07_rar/train_rar_baseline.py'
    print('Đang cập nhật repository Benchmark (git pull) để nạp mã nguồn mới nhất...')
    subprocess.run(f'git -C {BENCHMARK_DIR} pull', shell=True)
    if not os.path.exists(TRAIN_SCRIPT):
        TRAIN_SCRIPT = f'{WORK_DIR}/train_rar_baseline.py'

    cmd = [
        'torchrun', '--nproc_per_node=2', '--master_port=29541',
        TRAIN_SCRIPT,
        '--llama-repo',    LLAMA_DIR,
        '--pretrained',    GPT_CKPT_PATH,
        '--codes-dir',     CODES_DIR,
        '--model',         'GPT-B',
        '--epochs',        '10.0',
        '--batch-size',    '8',
        '--grad-accum',    '2',
        '--lr',            '5e-5',
        '--min-lr',        '1e-6',
        '--warmup-steps',  '100',
        '--out-dir',       CKPT_DIR,
        '--save-every',    '5',
        '--log-every',     '50'
    ]

    print('=' * 70)
    print('🚀 Bắt đầu HUẤN LUYỆN 10 Epochs cho RAR Thuần (Flat AR N=16,384) trên 2x GPU T4...')
    print('⚡ Cấu hình chuẩn vàng: Pure Float32 Precision + Gradient Checkpointing (BS=8x2x2, 3,120 steps)')
    print(' '.join(cmd))
    print('=' * 70)

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()

    if proc.returncode == 0:
        print('\n✅ HUẤN LUYỆN 10 EPOCHS CHO RAR THUẦN HOÀN TẤT THÀNH CÔNG!')
    else:
        raise RuntimeError(f'❌ Lỗi trong quá trình huấn luyện! Mã thoát: {proc.returncode}')


In [ ]:
# ============================================================
# CELL 6 — SINH 1,000 ẢNH MẪU TỪ RAR THUẦN (10-CLASS CÂN BẰNG)
# ============================================================
import os, subprocess, glob, sys, shutil

WORK_DIR          = '/kaggle/working'
LLAMA_DIR         = f'{WORK_DIR}/LlamaGen'
BENCHMARK_DIR     = f'{WORK_DIR}/Benchmark'
WEIGHTS_DIR       = f'{WORK_DIR}/weights'
CKPT_DIR          = f'{WORK_DIR}/checkpoints_rar_baseline'
SAMPLE_NPZ        = f'{WORK_DIR}/samples_rar_baseline_1000.npz'

USER_CKPT_PATH = ''

CKPT_CANDIDATES = [
    USER_CKPT_PATH,
    f'{CKPT_DIR}/ckpt_rar_baseline_10class_final.pt',
] + glob.glob('/kaggle/input/**/ckpt_rar_baseline_10class_final.pt', recursive=True) + [
    f'{WEIGHTS_DIR}/c2i_B_256.pt'  # Pretrained weights fallback
]

CKPT_PATH = next((p for p in CKPT_CANDIDATES if p and os.path.exists(p) and os.path.getsize(p) > 10000000), f'{CKPT_DIR}/ckpt_rar_baseline_10class_final.pt')

print('=' * 70)
print(f'✅ Sử dụng Checkpoint RAR Thuần: {CKPT_PATH}')
print(f'   -> Dung lượng: {os.path.getsize(CKPT_PATH)/1e6:.1f} MB')
print('=' * 70)

INFER_SCRIPT = f'{BENCHMARK_DIR}/baselines/07_rar/rar_baseline_infer.py'
print('Đang cập nhật repository Benchmark (git pull) để tải mã nguồn mới nhất...')
subprocess.run(f'git -C {BENCHMARK_DIR} pull', shell=True)
if not os.path.exists(INFER_SCRIPT):
    INFER_SCRIPT = f'{WORK_DIR}/rar_baseline_infer.py'

cmd = [
    'python', INFER_SCRIPT,
    '--llama-repo',    LLAMA_DIR,
    '--vq-ckpt',       f'{WEIGHTS_DIR}/vq_ds16_c2i.pt',
    '--ckpt',          CKPT_PATH,
    '--model',         'GPT-B',
    '--num-samples',   '1000',
    '--batch-size',    '32',
    '--cfg',           '2.25',
    '--temperature',   '1.0',
    '--top-k',         '1000',
    '--top-p',         '1.0',
    '--out',           SAMPLE_NPZ
]

print('🚀 Đang sinh 1,000 ảnh với KV-cache tăng tốc (~90s)...')
subprocess.run(' '.join(cmd), shell=True, check=True)


In [ ]:
# ============================================================
# CELL 7 — ĐÁNH GIÁ CHỈ SỐ: FID, INCEPTION SCORE, PRECISION, RECALL
# ============================================================
import os, sys, subprocess, numpy as np

WORK_DIR      = '/kaggle/working'
LLAMA_DIR     = f'{WORK_DIR}/LlamaGen'
SAMPLE_NPZ    = f'{WORK_DIR}/samples_rar_baseline_1000.npz'
REF_NPZ       = f'{WORK_DIR}/ref_batch_1000.npz'
EVAL_DIR      = f'{LLAMA_DIR}/evaluations/c2i'

print('=' * 70)
print('📊 ĐÁNH GIÁ CHỈ SỐ CHẤT LƯỢNG SINH ẢNH: RAR THUẦN (FLAT AR N=16,384)')
print('=' * 70)

eval_cmd = [
    'python', f'{EVAL_DIR}/evaluator.py',
    '--ref-batch', REF_NPZ,
    '--sample-batch', SAMPLE_NPZ
]
try:
    subprocess.run(' '.join(eval_cmd), shell=True, check=True)
except Exception as e:
    print(f'[Fallback Evaluation] Sử dụng torch-fidelity / scipy evaluator: {e}')
    from cleanfid import fid
    score = fid.compute_fid(SAMPLE_NPZ, REF_NPZ)
    print(f'FID: {score:.4f}')
